<a href="https://colab.research.google.com/github/JudahNdivo/bioinformatics-group-project/blob/main/notebooks/bioinformatics_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparative Analysis of Genetic Conservation and Variation in Selected SARS-CoV-2 Genes

In [1]:

from pathlib import Path
import os
import subprocess

# Detect Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/bioinformatics-group-project")

    # Clone GitHub repository if it is not already available
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git", "clone",
                "https://github.com/JudahNdivo/bioinformatics-group-project.git",
                str(PROJECT_ROOT)
            ],
            check=True
        )

    os.chdir(PROJECT_ROOT)

else:
    PROJECT_ROOT = Path(
        r"C:\Users\Judah\Desktop\bioinformatics-group-project"
    )

DATA_DIR = PROJECT_ROOT / "data"

print("Environment:", "Google Colab" if IN_COLAB else "VS Code")
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Data exists:", DATA_DIR.exists())
print("Reference exists:", (DATA_DIR / "wuhan_reference.fasta").exists())
print("Sequences exist:", (DATA_DIR / "sequences.fasta.xz").exists())
print("Metadata exists:", (DATA_DIR / "metadata.tsv.xz").exists())


Environment: VS Code
Project root: C:\Users\Judah\Desktop\bioinformatics-group-project
Data directory: C:\Users\Judah\Desktop\bioinformatics-group-project\data
Data exists: True
Reference exists: True
Sequences exist: True
Metadata exists: True


1. Research Question and Objectives

2. Data Acquisition:

The study uses publicly available SARS-CoV-2 genome sequences obtained from the National Center for Biotechnology Information (NCBI Virus/GenBank).

The dataset will contain approximately 20–30 complete genomes from each of four variants: Alpha, Beta, Delta and Omicron. The Wuhan-Hu-1 reference genome (NC_045512.2) will be included for comparison.

The selected genes are Spike (S), Envelope (E), Membrane (M), Nucleocapsid (N) and RNA-dependent RNA polymerase (RdRp).

Genome sequences will be stored in FASTA format, with accompanying metadata recording accession number, collection date, country and Pango lineage. The downloaded data will be passed to the preprocessing stage for quality checks and gene extraction.

Reference Genome Acquisition:

The Wuhan-Hu-1 reference genome (NC_045512.2) was obtained from NCBI GenBank using Biopython's Entrez module.

The reference genome contains 29,903 nucleotides and will serve as the baseline for comparing genetic variation across the selected SARS-CoV-2 variants.

The sequence was saved in FASTA format as `data/wuhan_reference.fasta` and successfully loaded using Biopython's SeqIO module.

In [2]:
%pip -q install biopython pandas

from Bio import SeqIO
from pathlib import Path

# Use the data directory defined in the first setup cell
reference_accession = "NC_045512.2"
reference_path = DATA_DIR / "wuhan_reference.fasta"

# Check that the reference genome exists
if not reference_path.exists():
    raise FileNotFoundError(
        f"Reference genome not found: {reference_path}. "
        "Ensure the project repository and data files are available."
    )

# Load the existing reference genome
reference = SeqIO.read(reference_path, "fasta")

# Verify the reference genome
print("Reference accession:", reference.id)
print("Genome length:", len(reference.seq))
print("Saved to:", reference_path.resolve())

# Confirm the expected accession and length
assert reference.id == reference_accession, "Incorrect reference accession"
assert len(reference.seq) == 29903, "Unexpected genome length"

print("Reference genome verification successful!")



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Reference accession: NC_045512.2
Genome length: 29903
Saved to: C:\Users\Judah\Desktop\bioinformatics-group-project\data\wuhan_reference.fasta
Reference genome verification successful!


In [3]:
from pathlib import Path
from Bio import SeqIO

# Find the project directory
cwd = Path.cwd()

if (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Project data folder not found. "
        "Open or clone the bioinformatics-group-project repository."
    )

DATA_DIR = PROJECT_ROOT / "data"

# Verify the reference genome
reference_path = DATA_DIR / "wuhan_reference.fasta"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Reference exists:", reference_path.exists())

if reference_path.exists():
    reference = SeqIO.read(reference_path, "fasta")
    print("Accession:", reference.id)
    print("Genome length:", len(reference.seq))


Project root: c:\Users\Judah\Desktop\bioinformatics-group-project
Data directory: c:\Users\Judah\Desktop\bioinformatics-group-project\data
Reference exists: True
Accession: NC_045512.2
Genome length: 29903


Sequence Metadata:

Metadata will be used to document the SARS-CoV-2 genome sequences included in the study.

Each record will contain its NCBI accession number, variant classification, Pango lineage, collection date, country of collection and sequence length.

This information will support data tracking, quality control and comparison of genetic variation across the selected variants.

In [4]:
import pandas as pd

# Define the metadata columns
columns = [
    "accession",
    "variant",
    "pango_lineage",
    "collection_date",
    "country",
    "sequence_length",
    "sequence_file"
]

# Create an empty metadata template
metadata = pd.DataFrame(columns=columns)

# Save inside the project's data folder
metadata_path = DATA_DIR / "metadata_template.csv"
metadata.to_csv(metadata_path, index=False)

print("Metadata template created:", metadata_path)
print("Columns:", metadata.columns.tolist())
print("Number of records:", len(metadata))

display(metadata)


Metadata template created: c:\Users\Judah\Desktop\bioinformatics-group-project\data\metadata_template.csv
Columns: ['accession', 'variant', 'pango_lineage', 'collection_date', 'country', 'sequence_length', 'sequence_file']
Number of records: 0


,accession,variant,pango_lineage,collection_date,country,sequence_length,sequence_file


Dataset File Description:

The project organizes biological sequence data and associated metadata in the `data/` directory.

| File | Description | Format |
|---|---|---|
| `wuhan_reference.fasta` | Wuhan-Hu-1 reference genome (NC_045512.2), containing 29,903 nucleotides | FASTA |
| `metadata_template.csv` | Empty metadata template containing seven fields for documenting biological sequences | CSV |
| Variant genome files | Genome sequences for Alpha, Beta, Delta and Omicron, to be added after acquisition | FASTA |
| Variant metadata | Accession numbers, lineage classifications, collection dates and countries, to be added after acquisition | CSV |

The reference genome provides a baseline for comparing genetic variation. The metadata table will link each acquired sequence to its source information using the accession number.

In [5]:

from pathlib import Path
import pandas as pd
from Bio import SeqIO

# Use the DATA_DIR established
reference_path = DATA_DIR / "wuhan_reference.fasta"
metadata_path = DATA_DIR / "metadata_template.csv"

files = {
    "Reference genome": reference_path,
    "Metadata template": metadata_path
}

print("DATA ACQUISITION FILE CHECK")
print("-" * 40)

for description, path in files.items():
    print(f"{description}:")
    print(f"  Exists: {path.exists()}")
    print(f"  Location: {path}")

    if path.exists():
        print(f"  Size: {path.stat().st_size:,} bytes")

# Verify reference sequence
if reference_path.exists():
    reference = SeqIO.read(reference_path, "fasta")
    print("\nReference accession:", reference.id)
    print("Reference length:", len(reference.seq))

# Verify metadata structure
if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)
    print("\nMetadata columns:", len(metadata.columns))
    print("Metadata records:", len(metadata))


DATA ACQUISITION FILE CHECK
----------------------------------------
Reference genome:
  Exists: True
  Location: c:\Users\Judah\Desktop\bioinformatics-group-project\data\wuhan_reference.fasta
  Size: 30,999 bytes
Metadata template:
  Exists: True
  Location: c:\Users\Judah\Desktop\bioinformatics-group-project\data\metadata_template.csv
  Size: 87 bytes

Reference accession: NC_045512.2
Reference length: 29903

Metadata columns: 7
Metadata records: 0


Variant Dataset Acquisition and Metadata Loading:

The study will use publicly available SARS-CoV-2 genome sequences representing Alpha, Beta, Delta and Omicron variants.

The genome sequences and their corresponding metadata will be obtained from NCBI Virus/GenBank. Each record will be identified by its accession number and associated with its variant, Pango lineage, collection date and country.

The acquired sequences will be loaded into Python using Biopython, while the metadata will be processed using pandas. These files will provide the input for subsequent preprocessing, quality control and comparative sequence analysis.

In [6]:
from pathlib import Path
import os

# Only run Google Colab setup inside Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/bioinformatics-group-project")

    if not PROJECT_ROOT.exists():
        !git clone https://github.com/JudahNdivo/bioinformatics-group-project.git

    os.chdir(PROJECT_ROOT)
    print("Running in Google Colab")
else:
    print("Running locally in VS Code")

Running locally in VS Code


In [7]:

import pandas as pd
import lzma

# Use the actual Nextstrain metadata file
metadata_file = DATA_DIR / "metadata.tsv.xz"

if not metadata_file.exists():
    raise FileNotFoundError(
        f"Metadata file not found: {metadata_file}"
    )

# Read compressed TSV metadata
with lzma.open(metadata_file, "rt", encoding="utf-8") as handle:
    metadata_df = pd.read_csv(
        handle,
        sep="\t",
        low_memory=False
    )

print("Total metadata records:", len(metadata_df))
print("Total metadata columns:", len(metadata_df.columns))

display(metadata_df.head())


Total metadata records: 5097
Total metadata columns: 56


,strain,virus,gisaid_epi_isl,genbank_accession,genbank_accession_rev,sra_accession,date,region,country,division,...,QC_frame_shifts,QC_stop_codons,QC_overall_score,QC_overall_status,frame_shifts,deletions,insertions,substitutions,aaSubstitutions,clock_deviation
0,AUS/VIC1749/2020,ncov,EPI_ISL_456642,MT612321,MT612321.1,SRR11915905,2020-05-25,Oceania,Australia,Victoria,...,good,good,25.757545,good,NaN,NaN,NaN,"C241T,C3037T,C3634T,C12488T,C14408T,C15324T,G1...","ORF1a:P4075S,ORF1b:P314L,ORF1b:E2072K,S:D614G",0.0
1,AUT/2021011795/2021,ncov,?,MW723194,MW723194.1,NaN,2021-03-01,Europe,Austria,Upper Austria,...,good,good,0.000000,good,NaN,"11288-11296,21765-21770,21992-21994,28271",NaN,"C241T,C913T,C3037T,C3267T,C5388A,C5986T,T6954C...","N:D3L,N:R203K,N:G204R,N:S235F,ORF1a:T1001I,ORF...",-3.0
2,AsiaticLion/IND/SARS-CoV-2_Lion_IND_CAD932_202...,ncov,?,MZ413340,MZ413340.1,NaN,2021-05,Asia,India,India,...,good,good,25.173611,good,NaN,NaN,NaN,"T65A,G210T,C241T,C1191T,C1267T,C3037T,C5184T,C...","M:I82T,N:D63G,N:L139F,N:R203M,N:D377Y,N:R385K,...",?
3,Atelesfusciceps/ECU/Atl.14_feces/2022,ncov,?,OQ630006,OQ630006.1,NaN,2022-09-20,South America,Ecuador,Ecuador,...,good,good,222.690415,bad,NaN,"21633-21641,21765-21770,28362-28370,29734-29759",NaN,"C2790T,C3037T,G4184A,C4321T,C9344T,A9424G,C953...","E:T9I,M:D3N,M:Q19E,M:A63T,N:P13L,N:R203K,N:G20...",-20.0
4,BEL/HT-USAFSAM-S20831/2022,ncov,EPI_ISL_16452070,OQ242023,OQ242023.1,SRR23060763,2022-11-07,Europe,Belgium,Hainaut,...,good,good,2.777778,good,NaN,"11288-11296,21633-21641,21765-21770,28362-2837...",NaN,"C241T,T670G,G1141A,T1570C,C2790T,C3037T,C3885T...","E:T9I,M:D3N,M:Q19E,M:A63T,N:P13L,N:R203K,N:G20...",6.0


In [8]:
from pathlib import Path
import lzma

metadata_file = DATA_DIR / "metadata.tsv.xz"
sequence_file = DATA_DIR / "sequences.fasta.xz"

print("NEXTSTRAIN DATASET VERIFICATION")
print("-" * 40)

for file in [metadata_file, sequence_file]:
    print(f"File: {file.name}")
    print(f"Exists: {file.exists()}")
    print(f"Size: {file.stat().st_size:,} bytes" if file.exists() else "Missing")

    if file.exists():
        try:
            with lzma.open(file, "rt", encoding="utf-8") as handle:
                first_line = handle.readline().strip()
            print("Compressed file readable: True")
            print("First line:", first_line[:150])
        except (lzma.LZMAError, EOFError, UnicodeError) as error:
            print("Error reading file:", error)

    print()

NEXTSTRAIN DATASET VERIFICATION
----------------------------------------
File: metadata.tsv.xz
Exists: True
Size: 539,904 bytes
Compressed file readable: True
First line: strain	virus	gisaid_epi_isl	genbank_accession	genbank_accession_rev	sra_accession	date	region	country	division	location	region_exposure	country_exposu

File: sequences.fasta.xz
Exists: True
Size: 554,944 bytes
Compressed file readable: True
First line: >RUS/Primorye_8507/2021



In [9]:
import pandas as pd
import lzma

with lzma.open(metadata_file, "rt", encoding="utf-8") as handle:
    metadata_columns = pd.read_csv(
        handle,
        sep="\t",
        nrows=0
    ).columns.tolist()

print("Number of metadata columns:", len(metadata_columns))

print("\nAvailable columns:")
for column in metadata_columns:
    print("-", column)

Number of metadata columns: 56

Available columns:
- strain
- virus
- gisaid_epi_isl
- genbank_accession
- genbank_accession_rev
- sra_accession
- date
- region
- country
- division
- location
- region_exposure
- country_exposure
- division_exposure
- segment
- length
- host
- age
- sex
- Nextstrain_clade
- pango_lineage
- GISAID_clade
- originating_lab
- submitting_lab
- authors
- url
- title
- paper_url
- date_submitted
- date_updated
- sampling_strategy
- database
- clade_nextstrain
- clade_who
- Nextclade_pango
- missing_data
- divergence
- nonACGTN
- coverage
- rare_mutations
- reversion_mutations
- potential_contaminants
- QC_missing_data
- QC_mixed_sites
- QC_rare_mutations
- QC_snp_clusters
- QC_frame_shifts
- QC_stop_codons
- QC_overall_score
- QC_overall_status
- frame_shifts
- deletions
- insertions
- substitutions
- aaSubstitutions
- clock_deviation


In [10]:
import pandas as pd
import lzma

# Load the existing Nextstrain metadata
with lzma.open(metadata_file, "rt", encoding="utf-8") as handle:
    metadata_df = pd.read_csv(handle, sep="\t", low_memory=False)

print("Total metadata records:", len(metadata_df))
print("Total columns:", len(metadata_df.columns))

# Inspect selected fields
selected_columns = [
    "strain",
    "genbank_accession",
    "pango_lineage",
    "clade_who",
    "date",
    "country",
    "length"
]

display(metadata_df[selected_columns].head())

print("\nMissing values:")
print(metadata_df[selected_columns].isna().sum())


Total metadata records: 5097
Total columns: 56


,strain,genbank_accession,pango_lineage,clade_who,date,country,length
0,AUS/VIC1749/2020,MT612321,B.1.210,NaN,2020-05-25,Australia,29797
1,AUT/2021011795/2021,MW723194,B.1.1.7,Alpha,2021-03-01,Austria,29763
2,AsiaticLion/IND/SARS-CoV-2_Lion_IND_CAD932_202...,MZ413340,B.1.617.2,Delta,2021-05,India,29874
3,Atelesfusciceps/ECU/Atl.14_feces/2022,OQ630006,BA.5,Omicron,2022-09-20,Ecuador,29425
4,BEL/HT-USAFSAM-S20831/2022,OQ242023,BA.5.9,Omicron,2022-11-07,Belgium,29683



Missing values:
strain                  0
genbank_accession       0
pango_lineage           0
clade_who            3537
date                    0
country                 0
length                  0
dtype: int64


Initial Metadata Inspection:

The Nextstrain metadata dataset contains 5,097 records and 56 columns.

Initial inspection showed that the accession number, Pango lineage, collection date, country and sequence length fields had no missing values. However, 3,537 records had missing WHO variant classifications in the `clade_who` column.

The dataset includes records labelled Alpha, Delta and Omicron in the initial sample, demonstrating the availability of historical variant information. Some records also originate from nonhuman hosts.

Further assessment is required to confirm coverage of all four study variants, the availability of matching nucleotide sequences and the suitability of the records for the planned comparative analysis.

In [11]:
import lzma

sequence_file = DATA_DIR / "sequences.fasta.xz"

sequence_count = 0
sample_headers = []

with lzma.open(sequence_file, "rt", encoding="utf-8") as handle:
    for line in handle:
        if line.startswith(">"):
            sequence_count += 1
            if len(sample_headers) < 5:
                sample_headers.append(line.strip())

print("Total FASTA records:", sequence_count)

print("\nFirst five FASTA headers:")
for header in sample_headers:
    print(header)

Total FASTA records: 5097

First five FASTA headers:
>RUS/Primorye_8507/2021
>USA/NE-SALIVA_S576105/2022
>IND/NGS-1491/2023
>USA/SC2-20261103-014/2026
>USA/2026CV0423/2026


In [12]:
import lzma
from Bio import SeqIO

# Load identifiers from the existing FASTA file
with lzma.open(sequence_file, "rt", encoding="utf-8") as handle:
    fasta_ids = {
        record.id for record in SeqIO.parse(handle, "fasta")
    }

# Load strain identifiers from metadata
metadata_ids = set(metadata_df["strain"].dropna().astype(str))

matching_ids = fasta_ids & metadata_ids

print("FASTA identifiers:", len(fasta_ids))
print("Metadata strain identifiers:", len(metadata_ids))
print("Matching identifiers:", len(matching_ids))
print("FASTA identifiers without metadata:", len(fasta_ids - metadata_ids))
print("Metadata identifiers without FASTA:", len(metadata_ids - fasta_ids))

FASTA identifiers: 5097
Metadata strain identifiers: 5097
Matching identifiers: 5097
FASTA identifiers without metadata: 0
Metadata identifiers without FASTA: 0


Sequence and Metadata Loading:

The study uses publicly available SARS-CoV-2 sequence data and metadata obtained from Nextstrain's open-data resources.

Two compressed files were downloaded:

- `sequences.fasta.xz` — nucleotide sequences in FASTA format.
- `metadata.tsv.xz` — associated metadata in tab-separated format.

Both files contain 5,097 records. The sequence identifiers were compared with the metadata `strain` column, and all 5,097 unique identifiers matched successfully.

The data is loaded using Python's `lzma` module, Biopython and pandas. The original files are retained without modification to preserve data provenance.

The dataset includes records labelled Alpha, Beta, Delta and Omicron, although variant representation is unequal and some records lack WHO classification labels.

In [13]:
# Inspect the available WHO variant labels
print("WHO variant label counts:")
print(metadata_df["clade_who"].value_counts(dropna=False))

print("\nHost categories:")
print(metadata_df["host"].value_counts(dropna=False).head(10))

WHO variant label counts:
clade_who
NaN            3537
Omicron        1105
recombinant     229
Delta           139
Alpha            49
Beta             19
Gamma            11
Eta               5
Epsilon           2
Lambda            1
Name: count, dtype: int64

Host categories:
host
Homo sapiens              4883
?                          190
NaN                         13
Panthera leo persica         2
Canis lupus familiaris       2
Ateles fusciceps             1
Felis catus                  1
Homo                         1
Mustela lutreola             1
Neogale vison                1
Name: count, dtype: int64


Variant Representation and Host Distribution:

The downloaded dataset contains records for all four target SARS-CoV-2 variants: Alpha (49), Beta (19), Delta (139) and Omicron (1,105). However, the distribution is uneven, and 3,537 records lack a WHO variant label.

Most records (4,883) are identified as human samples, while others have unknown or nonhuman hosts.

These findings indicate that additional quality control and assessment of variant representation are necessary before comparative sequence analysis.

3. Preprocessing and Quality Control

4. Exploratory Data Analysis 

5. Core Analysis

6. Biological Interpretation


7. Machine Learning

8. Conclusions and Limitations